# Topic-Matched False-to-True VAD Analysis

This notebook compares VAD scores between fake news articles from the existing `data/FakeVsTrueVAD/Fake.csv` file and LLM rewrites that attempt to make those articles truthful while preserving the original topic.

Flow:
1) load the existing fake-news CSV through the project dataset loader
2) prepare pair ids and original-text columns for traceability
3) rewrite each fake article into a truthful same-topic article with an LLM
4) score original and rewritten texts with the existing VAD model
5) compute paired deltas and topic-level summaries
6) export rewrite and VAD result CSVs

## 1) Imports

In [ ]:
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from misinformation_simulation.audits import (
    DEFAULT_VAD_MODEL_NAME,
    VAD_DIMENSIONS,
    annotate_vad_scores,
    build_rewrite_prompt,
    build_topic_matched_seed_dataset,
    compute_paired_vad_deltas,
    load_huggingface_vad_model,
    prepare_long_vad_frame,
    summarize_paired_vad_deltas,
    validate_rewritten_pairs,
)
from misinformation_simulation.datasets import load_fake_news_dataset
from misinformation_simulation.enums import Provider
from misinformation_simulation.llm.clients import create_llm_client, normalize_provider
from misinformation_simulation.llm.rate_limit import MinuteRateLimiter
from misinformation_simulation.llm.retry import (
    generate_gemini_text_with_retry,
    generate_openai_text_with_retry,
)

## 2) Configuration

In [ ]:
PROJECT_ROOT = Path("..").resolve()
SOURCE_DATA_DIR = PROJECT_ROOT / "data" / "FakeVsTrueVAD"
SOURCE_FAKE_CSV = SOURCE_DATA_DIR / "Fake.csv"
DATA_DIR = PROJECT_ROOT / "data" / "topic_matched_vad"
OUTPUT_DIR = PROJECT_ROOT / "output" / "audit" / "TopicMatchedVADAudit"

TEXT_COLUMN = "article_text"
TOPIC_COLUMN = "subject"
PAIR_COLUMN = "pair_id"

REWRITTEN_PATH = DATA_DIR / "false_news_rewritten_as_true.csv"
LONG_SCORED_PATH = OUTPUT_DIR / "topic_matched_vad_scored_long.csv"
PAIR_DELTAS_PATH = OUTPUT_DIR / "topic_matched_vad_pair_deltas.csv"
GLOBAL_SUMMARY_PATH = OUTPUT_DIR / "topic_matched_vad_global_summary.csv"
TOPIC_SUMMARY_PATH = OUTPUT_DIR / "topic_matched_vad_topic_summary.csv"

MODEL_NAME = DEFAULT_VAD_MODEL_NAME
BATCH_SIZE = 16
SHOW_PROGRESS = True

# Keep this small for the first paid/API run, then set to None for the full Fake.csv file.
MAX_FAKE_ROWS = 25
RANDOM_STATE = 42

# LLM rewrite settings. Supported providers follow the project Provider enum.
REWRITE_PROVIDER = Provider.GEMINI
REWRITE_MODEL = "gemini-2.5-flash-lite"
REWRITE_TEMPERATURE = 0.2
REWRITE_MAX_ATTEMPTS = 5
REWRITE_SLEEP_SECONDS = 0.0
REWRITE_MAX_REQUESTS_PER_MINUTE = None

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / ".env")

print(f"SOURCE_FAKE_CSV: {SOURCE_FAKE_CSV}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## 3) Load the fake-news source CSV

In [ ]:
fake_news_df = load_fake_news_dataset(
    SOURCE_DATA_DIR,
    output_text_column=TEXT_COLUMN,
    max_rows=MAX_FAKE_ROWS,
    random_state=RANDOM_STATE,
)

seed_df = build_topic_matched_seed_dataset(
    fake_news_df,
    text_column=TEXT_COLUMN,
    topic_column=TOPIC_COLUMN,
    pair_column=PAIR_COLUMN,
)

overview = (
    seed_df.groupby(TOPIC_COLUMN, dropna=False)
    .agg(articles=(PAIR_COLUMN, "size"), avg_words=("article_word_count", "mean"))
    .sort_values("articles", ascending=False)
    .reset_index()
)

display(overview)
display(seed_df[[PAIR_COLUMN, TOPIC_COLUMN, "title", "original_article_text"]].head())
print(f"Loaded {len(seed_df)} fake-news rows from: {SOURCE_FAKE_CSV}")

## 4) Inspect rewrite prompts before calling the LLM

In [ ]:
prompt_preview = build_rewrite_prompt(seed_df.iloc[0], topic_column=TOPIC_COLUMN)
print(prompt_preview[:3000])

## 5) Rewrite false articles as truthful same-topic articles

In [ ]:
REWRITE_SYSTEM_INSTRUCTION = (
    "You are a careful news verification and rewriting assistant. "
    "Rewrite false or unsupported news text into a truthful, neutral news "
    "article about the same topic. "
    "Preserve the approximate length, structure, and journalistic style. "
    "Correct or remove unsupported claims. Do not invent sources, "
    "quotes, numbers, dates, or events. "
    "Return only the rewritten article text."
)


def rewrite_article(prompt: str, *, client, provider: str, limiter: MinuteRateLimiter) -> str:
    if provider == "gemini":
        return generate_gemini_text_with_retry(
            client,
            model=REWRITE_MODEL,
            prompt=prompt,
            system_instruction=REWRITE_SYSTEM_INSTRUCTION,
            temperature=REWRITE_TEMPERATURE,
            max_attempts=REWRITE_MAX_ATTEMPTS,
            before_request_hook=limiter.acquire,
        )
    return generate_openai_text_with_retry(
        client,
        model=REWRITE_MODEL,
        prompt=prompt,
        system_instruction=REWRITE_SYSTEM_INSTRUCTION,
        temperature=REWRITE_TEMPERATURE,
        max_attempts=REWRITE_MAX_ATTEMPTS,
        before_request_hook=limiter.acquire,
    )


provider_name = normalize_provider(REWRITE_PROVIDER)
_, rewrite_client = create_llm_client(provider=provider_name)
limiter = MinuteRateLimiter(REWRITE_MAX_REQUESTS_PER_MINUTE)

if REWRITTEN_PATH.exists():
    rewritten_df = pd.read_csv(REWRITTEN_PATH)
    completed_pairs = set(
        rewritten_df.loc[rewritten_df["rewrite_status"].eq("success"), PAIR_COLUMN].astype(str)
    )
else:
    rewritten_df = seed_df.copy()
    rewritten_df["rewrite_provider"] = provider_name
    rewritten_df["rewrite_model"] = REWRITE_MODEL
    rewritten_df["rewrite_status"] = "not_requested"
    rewritten_df["rewrite_error"] = pd.NA
    rewritten_df["rewrite_prompt"] = pd.NA
    rewritten_df["rewritten_article_text"] = pd.NA
    completed_pairs = set()

for row_index, row in rewritten_df.iterrows():
    pair_id = str(row[PAIR_COLUMN])
    if pair_id in completed_pairs:
        continue

    prompt = build_rewrite_prompt(row, topic_column=TOPIC_COLUMN)
    rewritten_df.at[row_index, "rewrite_prompt"] = prompt
    rewritten_df.at[row_index, "rewrite_status"] = "running"
    rewritten_df.to_csv(REWRITTEN_PATH, index=False)

    try:
        rewritten_text = rewrite_article(
            prompt,
            client=rewrite_client,
            provider=provider_name,
            limiter=limiter,
        )
        rewritten_df.at[row_index, "rewritten_article_text"] = rewritten_text
        rewritten_df.at[row_index, "rewrite_status"] = "success"
        rewritten_df.at[row_index, "rewrite_error"] = pd.NA
    except Exception as exc:
        rewritten_df.at[row_index, "rewrite_status"] = "error"
        rewritten_df.at[row_index, "rewrite_error"] = str(exc)

    rewritten_df.to_csv(REWRITTEN_PATH, index=False)
    if REWRITE_SLEEP_SECONDS > 0:
        time.sleep(REWRITE_SLEEP_SECONDS)

rewrite_status_counts = rewritten_df["rewrite_status"].value_counts(dropna=False)
display(rewrite_status_counts)
display(
    rewritten_df[
        [PAIR_COLUMN, TOPIC_COLUMN, "rewrite_status", "title", "rewritten_article_text"]
    ].head()
)
print(f"Saved rewrite CSV to: {REWRITTEN_PATH}")

## 6) Validate completed rewrite pairs

In [ ]:
completed_rewrites_df = rewritten_df.loc[rewritten_df["rewrite_status"].eq("success")].copy()
validate_rewritten_pairs(completed_rewrites_df)

missing_rewrites = len(rewritten_df) - len(completed_rewrites_df)
if missing_rewrites:
    print(
        f"Warning: {missing_rewrites} rows are not successful rewrites "
        "and will be excluded from VAD."
    )

display(
    completed_rewrites_df[
        [PAIR_COLUMN, TOPIC_COLUMN, "title", "original_article_text", "rewritten_article_text"]
    ].head()
)

## 7) Score original and rewritten texts with VAD

In [ ]:
long_df = prepare_long_vad_frame(
    completed_rewrites_df,
    pair_column=PAIR_COLUMN,
    topic_column=TOPIC_COLUMN,
)

vad_model = load_huggingface_vad_model(model_name=MODEL_NAME)
scored_long_df = annotate_vad_scores(
    long_df,
    text_column=TEXT_COLUMN,
    model_bundle=vad_model,
    batch_size=BATCH_SIZE,
    show_progress=SHOW_PROGRESS,
    progress_description="Topic-matched VAD scoring",
)

scored_long_df.to_csv(LONG_SCORED_PATH, index=False)
display(scored_long_df.head())
print(f"Saved long scored CSV to: {LONG_SCORED_PATH}")

## 8) Compute paired deltas

In [ ]:
pair_delta_df = compute_paired_vad_deltas(
    scored_long_df,
    pair_column=PAIR_COLUMN,
    metadata_columns=("original_id", TOPIC_COLUMN, "title", "date"),
)
pair_delta_df.to_csv(PAIR_DELTAS_PATH, index=False)

display(pair_delta_df.head(10))
print(f"Saved pair deltas CSV to: {PAIR_DELTAS_PATH}")

## 9) Summaries

In [ ]:
global_summary_df = summarize_paired_vad_deltas(pair_delta_df)
topic_summary_df = summarize_paired_vad_deltas(pair_delta_df, group_column=TOPIC_COLUMN)

global_summary_df.to_csv(GLOBAL_SUMMARY_PATH, index=False)
topic_summary_df.to_csv(TOPIC_SUMMARY_PATH, index=False)

display(global_summary_df)
display(topic_summary_df)
print(f"Saved global summary to: {GLOBAL_SUMMARY_PATH}")
print(f"Saved topic summary to: {TOPIC_SUMMARY_PATH}")

## 10) Visual checks

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for axis, dimension in zip(axes, VAD_DIMENSIONS, strict=False):
        pair_delta_df[f"delta_{dimension}"].plot(kind="hist", bins=30, ax=axis)
        axis.axvline(0, color="black", linewidth=1)
        axis.set_title(f"Delta {dimension}: rewritten - false")
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for axis, dimension in zip(axes, VAD_DIMENSIONS, strict=False):
        axis.scatter(
            pair_delta_df[f"false_{dimension}"],
            pair_delta_df[f"rewritten_{dimension}"],
            alpha=0.6,
        )
        axis.set_xlabel(f"False {dimension}")
        axis.set_ylabel(f"Rewritten {dimension}")
        axis.set_title(dimension)
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib is not installed; skipping plots.")

## 11) Inspect largest changes

In [ ]:
columns_to_show = [
    PAIR_COLUMN,
    TOPIC_COLUMN,
    "title",
    "false_valence",
    "rewritten_valence",
    "delta_valence",
    "false_arousal",
    "rewritten_arousal",
    "delta_arousal",
    "false_dominance",
    "rewritten_dominance",
    "delta_dominance",
    "absolute_delta_sum",
]

display(pair_delta_df[columns_to_show].head(20))